### Record of the Lock-in signal during a piezo scan as a function of wavelength

In [ ]:
from pathlib import Path
import os
import sys

##### import project related moduls ####
current_file = Path.cwd() # cwd = path/*.ipynb - does not work in .py files.
print(f"current_file = {current_file}")
project_root = current_file.parent
print(f"project_root = {project_root}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from managers.lif_avp import LIFManager
import utils.scan_utils as su 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%reload_ext autoreload

In [ ]:
# connect all devises 
lm = LIFManager(silent=True)
lm.connect_all()

In [ ]:

lm.laser_on()

In [ ]:
## generate list of piezo voltages to scan
piezo_voltages = su.get_scan_list_stepped(
    min_val=-13.5, max_val=13.5, step=1,
    resolution=1E-3,
    reverse=False, zigzag=True
)
## piezo voltage return to 0
piezo_voltages.append(0)

print(f"piezo_voltages = {piezo_voltages}")
print(f"len(piezo_voltages) = {len(piezo_voltages)}")

In [ ]:
lm.wlm.average_off()

In [ ]:
lm.wlm.average_on()

In [ ]:
# lm.sleep_time = 0.01
print(lm.sleep_time)

In [ ]:
%matplotlib QtAgg
data = lm.scan_piezo(v_list=piezo_voltages, silent=True, life_plot=True)


In [ ]:
df = data["scan_data"]
df.head()

In [ ]:
lm.laser_off()
lm.disconnect_all()

### Plot Lock-in sigal vs wavelength

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
## to drow inline
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
plt.ioff()
initial_backend = plt.get_backend()
print(initial_backend)
try: 
    new_backend = plt.switch_backend('QtAgg')
    print(new_backend)
    df.plot(x="nm",
            y="Magnitude_V",
            marker='o',
            markersize=4,
            linestyle='-',
            )
    plt.show()
except Exception as e:
    print(f"Error: {e}")
finally:
    plt.switch_backend(initial_backend)


In [ ]:
print("hallo")

## Save data

In [ ]:
print(f"{lm.sleep_time=}")
print(f"{lm.wlm.time_sleep=}")

In [ ]:
import utils.file_utils as fu

In [ ]:
#### Make file name #####
data_dir = fu.make_data_dir(base_name="signal_hunt")
file_path = fu.make_data_file_name(
    data_dir=data_dir,
    base_name="ultra_fast_step0p1",
    extension="csv",
)
print(file_path)
####################################################

#### make meta data ##############################
meta_data = data["laser_state"]
comment = {
    "comment": "New power outlet, PMT:on, ",
    'lm.sleep_time (piezo)': lm.sleep_time,
}
meta_data.update(comment)
print(meta_data)
#################################################

#### save data ################################
print(type(data["scan_data"]))
df = data["scan_data"]
fu.save_dataframe(df=df,
                  file_path=file_path,
                  metadata=meta_data,
                  sep="\t",
                  index=True,
                  silent=False,
                  )
###############################################

In [ ]:
df.head()